## Fine-tuning distilGPT2 for Medical Dialogue Response Generation
### Decoder-only model with LoRA/PEFT for efficient training

This notebook fine-tunes the distilGPT2 model to act as a doctor responding to patients in medical dialogues.

Initially we wanted to finetune a 2B parameter model (Gemma2 2b), got hit by the weight of reality, quickly set in and pivoted to a real approach...

We tried encoder only models for text classification, and encoder-decoder models for text summarisation, all transformer self attention based, therefore. In order to finish the full experimentation, we will train a decoder only model that acts as the doctor and helps the incoming pacients.

The resulting model will be deployed in a raspberry pi home server, acting as a containerized server running in ollama, accesible via Fast-API from our personal portfolio pages and safely displayed into the web using cloudflared http/https tunnels. The domain will be the following: doctor.josuviteri.com (the domain is already available and waiting for our porgress) >:)

Our tasks were to use the MTS dialogs dataset in order to classify and summarise dialogs, therefore, in order to apply decoder only models, we will introduce token generation model as an extra task, inside of the personal interest experimentation that we mentioned in the last delivreable.

### Setup and Configuration

In [1]:
DATASET_PATH = "../../dataset/MTS-Dialog-TrainingSet.csv"

print(f"Dataset path: {DATASET_PATH}")

Dataset path: ../../dataset/MTS-Dialog-TrainingSet.csv


In [2]:
# CLEAN GPU MEMORY FROM PREVIOUS RUNS
import torch
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleared. Ready for training...")

Memory cleared. Ready for training...


In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import torch

/home/bnnyrabbit/Uni/4.o/NLP/Clinic-Note-NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-12 07:57:22.943793: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/bnnyrabbit/Uni/4.o/NLP/Clinic-Note-NLP/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Global Configuration

In [4]:
# Model configuration
MODEL_NAME = "distilgpt2"  # distilgpt2 82 M model

# Training hyperparameters
MAX_LENGTH = 256  
LEARNING_RATE = 5e-5
NUM_EPOCHS = 50
PER_DEVICE_BATCH_SIZE = 16  
GRADIENT_ACCUMULATION_STEPS = 1 

# LoRA configuration for efficient fine-tuning
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# Output directory
OUTPUT_DIR = "./distilGPT2_medical_dialogue"

SEED = 42
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### Data Loading and Preprocessing

In [5]:
# Load dataset
df = pd.read_csv(DATASET_PATH)
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

Dataset shape: (1201, 4)

Columns: ['ID', 'section_header', 'section_text', 'dialogue']

First few rows:
   ID section_header                                       section_text  \
0   0          GENHX  The patient is a 76-year-old white female who ...   
1   1          GENHX  The patient is a 25-year-old right-handed Cauc...   
2   2          GENHX  This is a 22-year-old female, who presented to...   
3   3    MEDICATIONS  Prescribed medications were Salmeterol inhaler...   
4   4             CC                                   Burn, right arm.   

                                            dialogue  
0  Doctor: What brings you back into the clinic t...  
1  Doctor: How're you feeling today?  \r\nPatient...  
2  Doctor: Hello, miss. What is the reason for yo...  
3  Doctor: Are you taking any over the counter me...  
4  Doctor: Hi, how are you? \r\nPatient: I burned...  


In [6]:
# Data preprocessing function
def clean_dialogue(text):
    """Clean and normalize dialogue text"""
    if pd.isna(text):
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_patient_doctor_turns(dialogue):
    """Extract patient and doctor turns from dialogue"""
    # Split dialogue into turns based on speaker labels
    pattern = r'(Patient|Doctor|Doctor_2|Guest_family(?:_\d)?|Guest_clinician):\s*'
    turns = re.split(pattern, dialogue, flags=re.IGNORECASE)
    
    conversations = []
    for i in range(1, len(turns), 2):
        if i+1 < len(turns):
            speaker = turns[i].strip()
            text = turns[i+1].strip()
            if text:
                conversations.append({"speaker": speaker, "text": text})
    return conversations

def format_conversation_for_training(dialogue):
    """Format dialogue for causal language modeling"""
    turns = extract_patient_doctor_turns(dialogue)
    
    training_examples = []
    context = []
    
    for turn in turns:
        speaker = turn["speaker"]
        text = turn["text"]
        
        # When we see a doctor response, create a training example
        if speaker.lower().startswith('doctor'):
            if context:
                # Format: <context> -> Doctor response
                prompt = " ".join(context)
                response = text
                training_examples.append({"prompt": prompt, "response": response})
        
        # context for next doctor response
        context.append(f"{speaker}: {text}")
    
    return training_examples

# Clean and prepare data
df['dialogue_clean'] = df['dialogue'].apply(clean_dialogue)

# Extract training examples
all_examples = []
for dialogue in df['dialogue_clean']:
    examples = format_conversation_for_training(dialogue)
    all_examples.extend(examples)

print(f"\nTotal training examples extracted: {len(all_examples)}")
print(f"\nExample training pair:")
if all_examples:
    print(f"Prompt: {all_examples[0]['prompt'][:200]}...")
    print(f"Response: {all_examples[0]['response'][:200]}...")


Total training examples extracted: 4708

Example training pair:
Prompt: Doctor: What brings you back into the clinic today, miss? Patient: I came in for a refill of my blood pressure medicine....
Response: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones. Have you noticed any changes o...


In [7]:
# Create train/test split
train_data, test_data = train_test_split(
    all_examples, test_size=0.2, random_state=SEED
)

print(f"Training examples: {len(train_data)}")
print(f"Test examples: {len(test_data)}")

Training examples: 3766
Test examples: 942


### Model and Tokenizer Loading

In [8]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Gemma models may not have a pad token, so we add one
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left" 

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")
print(f"PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

Tokenizer loaded. Vocab size: 50257
PAD token: <|endoftext|> (ID: 50256)
EOS token: <|endoftext|> (ID: 50256)


/home/bnnyrabbit/Uni/4.o/NLP/Clinic-Note-NLP/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:

bnb_config = None
print("Loading model in full precision")

Loading model in full precision


In [10]:
# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")


Model loaded: distilgpt2
Model device: cuda:0
Model dtype: torch.float16


### LoRA/PEFT Configuration

We use lora to fine tune the model with out changing mora than 1% of the weights. This approach is the best for our case, as we have low amount of data and hardware. This way we only change the adapters, but we avoid overfitting with our use case data. Mantaining the generalization as near to its original level.

In [11]:
# Configure LoRA
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["c_attn", "c_proj"],  # GPT-2 attention layers
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\nLoRA applied successfully!")

trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804

LoRA applied successfully!


/home/bnnyrabbit/Uni/4.o/NLP/Clinic-Note-NLP/.venv/lib/python3.12/site-packages/peft/tuners/lora/layer.py:1091: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


### Dataset Preparation and Tokenization

In [12]:
def format_instruction(example):
    """Format examples as instruction-following prompts"""
    instruction = "You are a medical doctor. Respond to the patient's concerns professionally and compassionately."
    
    # Format: <instruction> <context> <doctor response>
    # IMPORTANT: Include "Doctor:" prefix in both prompt and response for consistency
    text = f"""### Instruction:
{instruction}

### Context:
{example['prompt']}

### Response:
Doctor: {example['response']}{tokenizer.eos_token}"""
    
    return text

# Format and tokenize data
def tokenize_function(examples):
    # Format texts
    texts = [format_instruction(ex) for ex in examples]
    
    # Tokenize
    tokenized = tokenizer(
        texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    
    # For causal LM, labels are the same as input_ids
    tokenized["labels"] = tokenized["input_ids"].clone()
    
    return tokenized

In [13]:
# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

print(f"Train dataset: {train_dataset}")
print(f"Test dataset: {test_dataset}")

# Preview formatted example
print("\n" + "="*50)
print("Example formatted prompt:")
print("="*50)
print(format_instruction(train_data[0]))

Train dataset: Dataset({
    features: ['prompt', 'response'],
    num_rows: 3766
})
Test dataset: Dataset({
    features: ['prompt', 'response'],
    num_rows: 942
})

Example formatted prompt:
### Instruction:
You are a medical doctor. Respond to the patient's concerns professionally and compassionately.

### Context:
Doctor: Good morning, Miss X Y Z, correct? Patient: Yes, that's me, good morning doctor. Doctor: Before we begin, I just need a few pieces of information. How old are you? Patient: I'm forty four years young, doctor. Doctor: Good, thank you. Next, which hand do you write with? Patient: I write with my right hand. Doctor: Finally, what do you do for a living? Patient: I'm an aircraft mechanic. Doctor: Very nice, so, how did you get hurt? Patient: Um, I was working on repairing an airplane at work when I fell between the plane and one of the stands. Doctor: How big was the gap that you stepped in? Patient: Um, it was about a foot and a half. Doctor: Which knee did you hur

In [14]:
# Tokenize datasets
def preprocess_batch(examples):
    texts = [format_instruction(ex) for ex in examples]
    model_inputs = tokenizer(
        texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# Apply tokenization - this needs to work with dict format
def tokenize_dataset(dataset):
    texts = [format_instruction(ex) for ex in dataset]
    tokenized = tokenizer(
        texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
     
    # Create labels with masking for instruction/context
    labels = []
    for i, text in enumerate(texts):
        # Tokenize the full text
        full_ids = tokenized["input_ids"][i]
        
        # Find where "### Response:\nDoctor:" ends to start training from there
        response_text = "### Response:\nDoctor:"
        response_ids = tokenizer.encode(response_text, add_special_tokens=False)
        
        # Create label with -100 for prompt tokens (ignored in loss)
        label = [-100] * len(full_ids)
        
        # Find the response start position
        for j in range(len(full_ids) - len(response_ids)):
            if full_ids[j:j+len(response_ids)] == response_ids:
                # Start training from after "### Response:\nDoctor:"
                response_start = j + len(response_ids)
                label[response_start:] = full_ids[response_start:]
                break
        
        labels.append(label)
    
    tokenized["labels"] = labels
    return Dataset.from_dict(tokenized)

train_dataset_tokenized = tokenize_dataset(train_data)
test_dataset_tokenized = tokenize_dataset(test_data)

print("\nDatasets tokenized successfully!")
print(f"Train: {train_dataset_tokenized}")
print(f"Test: {test_dataset_tokenized}")
print("\nNote: Labels masked for instruction/context - only training on doctor responses!")
print("Training format now includes 'Doctor:' prefix for role consistency.")


Datasets tokenized successfully!
Train: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3766
})
Test: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 942
})

Note: Labels masked for instruction/context - only training on doctor responses!
Training format now includes 'Doctor:' prefix for role consistency.


### Training Configuration

In [15]:
# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

In [16]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, 
    learning_rate=LEARNING_RATE, 
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True, 
    optim="adamw_torch", 
    gradient_checkpointing=False,
    seed=SEED,
)


### Training

In [17]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=test_dataset_tokenized,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Trainer initialized. Ready to train!")

Trainer initialized. Ready to train!


/home/bnnyrabbit/Uni/4.o/NLP/Clinic-Note-NLP/.venv/lib/python3.12/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [18]:
# Start training
print("="*60)
print("Starting distilGPT2 fine-tuning for medical dialogue...")
print("="*60)

trainer.train()

Starting distilGPT2 fine-tuning for medical dialogue...


Epoch,Training Loss,Validation Loss
1,3.005000,2.758727
2,2.503000,2.330318
3,2.362200,2.221680
4,2.313000,2.155783
5,2.237900,2.105427
6,2.215000,2.066523
7,2.130800,2.034835
8,2.100400,2.001090
9,2.118600,1.971225
10,2.051900,1.947483


TrainOutput(global_step=11800, training_loss=1.9801014913138697, metrics={'train_runtime': 3132.4344, 'train_samples_per_second': 60.113, 'train_steps_per_second': 3.767, 'total_flos': 1.25351114047488e+16, 'train_loss': 1.9801014913138697, 'epoch': 50.0})

### Save Model

In [19]:
# Save the fine-tuned model
trainer.save_model(f"{OUTPUT_DIR}/final_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/final_model")

print(f"\nModel saved to {OUTPUT_DIR}/final_model")

# Save LoRA adapters separately
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapters")
print(f"LoRA adapters saved to {OUTPUT_DIR}/lora_adapters")


Model saved to ./distilGPT2_medical_dialogue/final_model
LoRA adapters saved to ./distilGPT2_medical_dialogue/lora_adapters


### Inference & Testing

In [20]:
# Test the model with a sample dialogue
def generate_doctor_response(context, max_new_tokens=150):
    """Generate a doctor's response given the conversation context"""
    instruction = "You are a medical doctor. Respond to the patient's concerns professionally and compassionately."
    
    prompt = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
Doctor:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Define stopping criteria to stop at speaker labels
    from transformers import StoppingCriteria, StoppingCriteriaList
    
    class StopOnSpeakerLabel(StoppingCriteria):
        def __init__(self, tokenizer, stop_strings):
            self.tokenizer = tokenizer
            self.stop_strings = stop_strings
            
        def __call__(self, input_ids, scores, **kwargs):
            # Decode the generated text so far
            generated_text = self.tokenizer.decode(input_ids[0], skip_special_tokens=True)
            # Check if any stop string appears
            for stop_str in self.stop_strings:
                if stop_str in generated_text.split("Doctor:")[-1]:
                    return True
            return False
    
    stopping_criteria = StoppingCriteriaList([
        StopOnSpeakerLabel(tokenizer, ["Patient:", "Guest_family:", "Guest_clinician:", "Doctor_2:"])
    ])
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,
        stopping_criteria=stopping_criteria
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the response part after "Doctor:"
    if "Doctor:" in response:
        response = response.split("Doctor:")[-1].strip()
    elif "### Response:" in response:
        response = response.split("### Response:")[-1].strip()
    
    # Remove any speaker labels that might have leaked through
    for speaker in ["Patient:", "Guest_family:", "Guest_clinician:", "Doctor_2:", "Doctor:"]:
        if speaker in response:
            response = response.split(speaker)[0].strip()
            break
    
    return response

In [21]:
# Test with a sample from the test set
test_example = test_data[0]

print("="*60)
print("TEST INFERENCE")
print("="*60)
print(f"\nContext:\n{test_example['prompt']}\n")
print(f"\nGround Truth Response:\n{test_example['response']}\n")
print("\nGenerated Response:")
print("-"*60)

generated = generate_doctor_response(test_example['prompt'])
print(generated)

TEST INFERENCE

Context:
Doctor: You have atrial fibrillation from the past? Patient: Yes, rhythm problem is bad. Doctor: And no dizziness? Patient: Yes. I do.


Ground Truth Response:
Okay well...


Generated Response:
------------------------------------------------------------
How old was your pregnancy? Do you know when or when they began?


In [22]:
# Interactive testing
def chat_with_doctor():
    """Interactive chat function to test the model"""
    print("\n" + "="*60)
    print("Medical Chatbot - Type 'quit' to exit")
    print("="*60 + "\n")
    
    conversation_history = []
    
    while True:
        user_input = input("Patient: ")
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Ending consultation. Take care!")
            break
        
        # Add to conversation history
        conversation_history.append(f"Patient: {user_input}")
        context = " ".join(conversation_history)
        
        # Generate response
        response = generate_doctor_response(context)
        
        conversation_history.append(f"Doctor: {response}")
        
        print(f"\nDoctor: {response}\n")

interactive chat

### Evaluation Metrics

In [23]:
# Evaluate on test set
print("\nEvaluating on test set...")
eval_results = trainer.evaluate()

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")


Evaluating on test set...



EVALUATION RESULTS
eval_loss: 1.6583
eval_runtime: 5.2958
eval_samples_per_second: 177.8760
eval_steps_per_second: 11.1410
epoch: 50.0000


### Loading the Fine-tuned Model Later

In [24]:
# To load fine-tuned model later:
# 
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from peft import PeftModel
#
# # Load base model
# base_model = AutoModelForCausalLM.from_pretrained(
#     "google/gemma-2-2b",
#     device_map="auto",
#     torch_dtype=torch.bfloat16
# )
#
# # Load LoRA adapters
# model = PeftModel.from_pretrained(base_model, "./distilGPT2_medical_dialogue/lora_adapters")
# tokenizer = AutoTokenizer.from_pretrained("./distilGPT2_medical_dialogue/final_model")
#
# # use model for inference